In [3]:
import pandas as pd

# Load the data
output_path = "Text Relevance Analysis Case View_052925.xlsx - Full Results 052925.csv"
df = pd.read_csv(output_path)
df.columns

correct_answer_path = "gpt5-relevancy-combined-dec-12.csv"
correct_df = pd.read_csv(correct_answer_path)
# correct_df.columns

In [4]:
# Merge the dataframes on Origin and ID_corr
merged = df.merge(
    correct_df[['ID_corr', 'answer_corr', 'data_source_corr']], 
    left_on='Origin', 
    right_on='ID_corr', 
    how='left'
)

# Add the answer_corr column to df
df['answer_corr'] = merged['answer_corr']
df['data_source_corr'] = merged['data_source_corr']

# Compare q1 with answer_corr (both converted to lowercase)
# If answer_corr is NaN (no match found), set Match to FALSE
df['Match'] = merged.apply(
    lambda row: 'TRUE' if pd.notna(row['answer_corr']) and 
                str(row['q1']).lower() == str(row['answer_corr']).lower() 
                else 'FALSE', 
    axis=1
)

In [5]:
df.head(50)

,Origin,Problem ID,Duration,Labeling state,# Qualified reads,User ID,q1,q2,q3,q4,...,q16,q17,q18,q19,q20,q21,q22,answer_corr,data_source_corr,Match
0,ID2000,25261945,84.593,Labeled,3,579501,c,high relevance,high relevance,high relevance,...,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,NaN,NaN,FALSE
1,ID2000,25261945,180.905,Labeled,3,579796,d,low relevance,high relevance,high relevance,...,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,NaN,NaN,FALSE
2,ID2000,25261945,72.242,Labeled,3,581648,a,high relevance,high relevance,high relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,FALSE
3,ID1999,25261946,292.634,Labeled,4,579918,e,low relevance,high relevance,low relevance,...,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,C,medxpert,FALSE
4,ID1999,25261946,203.529,Labeled,4,581304,e,high relevance,high relevance,low relevance,...,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,C,medxpert,FALSE
5,ID1999,25261946,226.492,Labeled,4,579536,e,low relevance,high relevance,low relevance,...,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,C,medxpert,FALSE
6,ID1999,25261946,217.128,Labeled,4,579516,c,high relevance,high relevance,low relevance,...,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,C,medxpert,TRUE
7,ID1998,25261947,302.918,Labeled,3,348391,d,low relevance,high relevance,high relevance,...,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,A,medbullets,FALSE
8,ID1998,25261947,175.523,Labeled,3,579501,a,not relevant,not relevant,not relevant,...,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,A,medbullets,TRUE
9,ID1998,25261947,168.096,Labeled,3,581648,a,high relevance,high relevance,high relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A,medbullets,TRUE


In [7]:
# Count unique Origins
unique_origins = df['Origin'].nunique()
print(f"Number of unique Origins: {unique_origins}")
print("\n" + "="*50 + "\n")

# Summary: Count how many Origins have 3, 2, 1, or 0 TRUE matches
print("="*50)
print("SUMMARY OF TRUE MATCHES PER ORIGIN:")
print("="*50)

true_counts = df[df['Match'] == 'TRUE'].groupby('Origin').size()
all_origins = df.groupby('Origin').size()

# Count origins by number of TRUE matches
origins_with_3_true = (true_counts == 3).sum()
origins_with_2_true = (true_counts == 2).sum()
origins_with_1_true = (true_counts == 1).sum()
origins_with_0_true = len(all_origins) - len(true_counts)

print(f"Origins with 3 TRUE matches: {origins_with_3_true}")
print(f"Origins with 2 TRUE matches: {origins_with_2_true}")
print(f"Origins with 1 TRUE match: {origins_with_1_true}")
print(f"Origins with 0 TRUE matches: {origins_with_0_true}")

# NEW ANALYSIS: Average total matches for Origins with >0 TRUE matches
print("\n" + "="*50)
print("AVERAGE TOTAL MATCHES FOR ORIGINS WITH >0 TRUE MATCHES:")
print("="*50)

# Get Origins with at least one TRUE match
origins_with_true = true_counts[true_counts > 0].index

# Calculate average total matches for these Origins
total_matches_per_origin = all_origins[origins_with_true]
avg_total_matches = total_matches_per_origin.mean()

print(f"\nOverall average total matches (for Origins with >0 TRUE): {avg_total_matches:.2f}")

# Break down by data_source_corr
print("\nBreakdown by data_source_corr:")
print("-" * 50)

for data_source in df['data_source_corr'].unique():
    # Filter for this data_source
    df_source = df[df['data_source_corr'] == data_source]
    
    # Get TRUE counts and total counts for this data_source
    true_counts_source = df_source[df_source['Match'] == 'TRUE'].groupby('Origin').size()
    all_origins_source = df_source.groupby('Origin').size()
    
    # Get Origins with at least one TRUE match
    origins_with_true_source = true_counts_source[true_counts_source > 0].index
    
    if len(origins_with_true_source) > 0:
        total_matches_source = all_origins_source[origins_with_true_source]
        avg_total_source = total_matches_source.mean()
        print(f"  {data_source}: {avg_total_source:.2f} (n={len(origins_with_true_source)} Origins)")
    else:
        print(f"  {data_source}: No Origins with TRUE matches")

# NEW ANALYSIS: Break down TRUE match counts by data_source_corr
# Create a table for TRUE match counts breakdown by data_source_corr
print("="*70)
print("TRUE MATCH COUNTS BREAKDOWN BY data_source_corr:")
print("="*70)

# Initialize lists to store data for the table
data_sources = []
origins_3_true_list = []
origins_2_true_list = []
origins_1_true_list = []
origins_0_true_list = []
total_origins_list = []

for data_source in df['data_source_corr'].unique():
    # Filter for this data_source
    df_source = df[df['data_source_corr'] == data_source]
    
    # Get TRUE counts for this data_source
    true_counts_source = df_source[df_source['Match'] == 'TRUE'].groupby('Origin').size()
    all_origins_source = df_source.groupby('Origin').size()
    
    # Count origins by number of TRUE matches
    origins_3_true = (true_counts_source == 3).sum()
    origins_2_true = (true_counts_source == 2).sum()
    origins_1_true = (true_counts_source == 1).sum()
    origins_0_true = len(all_origins_source) - len(true_counts_source)
    total_origins = len(all_origins_source)
    
    # Append to lists
    data_sources.append(data_source)
    origins_3_true_list.append(origins_3_true)
    origins_2_true_list.append(origins_2_true)
    origins_1_true_list.append(origins_1_true)
    origins_0_true_list.append(origins_0_true)
    total_origins_list.append(total_origins)

# Create DataFrame
summary_table = pd.DataFrame({
    'Data Source': data_sources,
    '3 TRUE Matches': origins_3_true_list,
    '2 TRUE Matches': origins_2_true_list,
    '1 TRUE Match': origins_1_true_list,
#     '0 TRUE Matches': origins_0_true_list,
    'Total Origins': total_origins_list
})

# Display the table
print(summary_table.to_string(index=False))

# Also add a total row
print("\n" + "-"*70)
totals = summary_table[['3 TRUE Matches', '2 TRUE Matches', '1 TRUE Match', 
                        'Total Origins']].sum()
print(f"{'TOTAL':<20} {totals['3 TRUE Matches']:>15} {totals['2 TRUE Matches']:>15} "
      f"{totals['1 TRUE Match']:>13} {totals['Total Origins']:>14}")


Number of unique Origins: 2000


SUMMARY OF TRUE MATCHES PER ORIGIN:
Origins with 3 TRUE matches: 467
Origins with 2 TRUE matches: 404
Origins with 1 TRUE match: 419
Origins with 0 TRUE matches: 700

AVERAGE TOTAL MATCHES FOR ORIGINS WITH >0 TRUE MATCHES:

Overall average total matches (for Origins with >0 TRUE): 3.03

Breakdown by data_source_corr:
--------------------------------------------------
  nan: No Origins with TRUE matches
  medxpert: 3.03 (n=318 Origins)
  medbullets: 3.01 (n=207 Origins)
  mmlu: 3.02 (n=193 Origins)
  jama: 3.04 (n=582 Origins)
TRUE MATCH COUNTS BREAKDOWN BY data_source_corr:
Data Source  3 TRUE Matches  2 TRUE Matches  1 TRUE Match  Total Origins
        NaN               0               0             0              0
   medxpert              35              75           208            318
 medbullets             101              67            38            207
       mmlu             126              53            12            193
       jama          

In [9]:
# Filter for TRUE matches only
df_true = df[df['Match'] == 'TRUE'].copy()

# Define the label columns
label_columns = ['q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 
                 'q12', 'q13', 'q14', 'q15', 'q16', 'q17', 'q18', 'q19', 'q20', 'q21', 'q22']

# Standardize the labels: map to High, Low, Irr
def standardize_label(val):
    if pd.isna(val):
        return None
    val_str = str(val).strip().lower()
    if 'high' in val_str:
        return 'High'
    elif 'low' in val_str:
        return 'Low'
    elif 'not relevant' in val_str or 'irr' in val_str:
        return 'Irr'
    else:
        return val_str

# Apply standardization to all label columns
for col in label_columns:
    if col in df_true.columns:
        df_true[col] = df_true[col].apply(standardize_label)

# Get Origins by number of TRUE matches
true_counts = df_true.groupby('Origin').size()
origins_with_3 = true_counts[true_counts == 3].index
origins_with_2 = true_counts[true_counts == 2].index
origins_with_1 = true_counts[true_counts == 1].index

print(f"Number of Origins with exactly 3 TRUE matches: {len(origins_with_3)}")
print(f"Number of Origins with exactly 2 TRUE matches: {len(origins_with_2)}")
print(f"Number of Origins with exactly 1 TRUE match: {len(origins_with_1)}")
print()

# Define all possible combinations for 3 correct labels
combinations_3_labels = {
    'High Relevance Labels': [
        ('High', 'High', 'High'),
        ('High', 'High', 'Low'),
        ('High', 'High', 'Irr'),
        ('High', 'Low', 'Low')
    ],
    'Low Relevance Labels': [
        ('Low', 'Low', 'Low'),
        ('High', 'Low', 'Irr')
    ],
    'Irrelevant Labels': [
        ('High', 'Irr', 'Irr'),
        ('Low', 'Low', 'Irr'),
        ('Low', 'Irr', 'Irr'),
        ('Irr', 'Irr', 'Irr')
    ]
}

# Define all possible combinations for 2 correct labels
combinations_2_labels = {
    'High Relevance Labels': [
        ('High', 'High'),
        ('High', 'Low')
    ],
    'Low Relevance Labels': [
        ('Low', 'Low'),
        ('High', 'Irr'),
        ('Low', 'Irr')
    ],
    'Irrelevant Labels': [
        ('Irr', 'Irr')
    ]
}

# Define all possible combinations for 1 correct label
combinations_1_label = {
    'High Relevance Labels': [
        ('High',)
    ],
    'Low Relevance Labels': [
        ('Low',)
    ],
    'Irrelevant Labels': [
        ('Irr',)
    ]
}

# Function to check if a set of labels matches a combination pattern
def matches_combination(labels, combination):
    """Check if the sorted labels match the sorted combination"""
    labels_sorted = tuple(sorted(labels))
    combo_sorted = tuple(sorted(combination))
    return labels_sorted == combo_sorted

# Function to analyze Origins with n TRUE matches
def analyze_origins(origins, n_matches, combinations_dict, df_data, label_cols):
    """Analyze label combinations for Origins with n TRUE matches"""
    df_filtered = df_data[df_data['Origin'].isin(origins)].copy()
    results = []
    
    for origin in origins:
        origin_data = df_filtered[df_filtered['Origin'] == origin]
        
        # For each label column, get the labels for this Origin
        for col in label_cols:
            if col in origin_data.columns:
                labels = origin_data[col].dropna().tolist()
                
                # Only process if we have exactly n labels
                if len(labels) == n_matches:
                    # Check which combination this matches
                    matched_category = None
                    matched_combination = None
                    
                    for category, combos in combinations_dict.items():
                        for combo in combos:
                            if matches_combination(labels, combo):
                                matched_category = category
                                matched_combination = combo
                                break
                        if matched_category:
                            break
                    
                    results.append({
                        'Origin': origin,
                        'Question': col,
                        'Labels': tuple(sorted(labels)),
                        'Category': matched_category if matched_category else 'Other',
                        'Combination': matched_combination if matched_combination else None
                    })
    
    return pd.DataFrame(results)

# Analyze Origins with 3 TRUE matches
print("="*80)
print("ANALYZING ORIGINS WITH 3 TRUE MATCHES")
print("="*80)
results_3 = analyze_origins(origins_with_3, 3, combinations_3_labels, df_true, label_columns)

if len(results_3) > 0:
    category_counts_3 = results_3['Category'].value_counts()
    print("\nCounts by Category:")
    print(category_counts_3)
    
    print("\nDetailed Combination Counts:")
    for category in ['High Relevance Labels', 'Low Relevance Labels', 'Irrelevant Labels', 'Other']:
        if category in results_3['Category'].values:
            cat_df = results_3[results_3['Category'] == category]
            print(f"\n{category}:")
            combo_counts = cat_df['Labels'].value_counts()
            for combo, count in combo_counts.items():
                print(f"  {combo}: {count}")
    
    results_3.to_csv('label_combinations_3_matches.csv', index=False)
    print("\n✓ Results saved to 'label_combinations_3_matches.csv'")

# Analyze Origins with 2 TRUE matches
print("\n" + "="*80)
print("ANALYZING ORIGINS WITH 2 TRUE MATCHES")
print("="*80)
results_2 = analyze_origins(origins_with_2, 2, combinations_2_labels, df_true, label_columns)

if len(results_2) > 0:
    category_counts_2 = results_2['Category'].value_counts()
    print("\nCounts by Category:")
    print(category_counts_2)
    
    print("\nDetailed Combination Counts:")
    for category in ['High Relevance Labels', 'Low Relevance Labels', 'Irrelevant Labels', 'Other']:
        if category in results_2['Category'].values:
            cat_df = results_2[results_2['Category'] == category]
            print(f"\n{category}:")
            combo_counts = cat_df['Labels'].value_counts()
            for combo, count in combo_counts.items():
                print(f"  {combo}: {count}")
    
    results_2.to_csv('label_combinations_2_matches.csv', index=False)
    print("\n✓ Results saved to 'label_combinations_2_matches.csv'")

# Analyze Origins with 1 TRUE match
print("\n" + "="*80)
print("ANALYZING ORIGINS WITH 1 TRUE MATCH")
print("="*80)
results_1 = analyze_origins(origins_with_1, 1, combinations_1_label, df_true, label_columns)

if len(results_1) > 0:
    category_counts_1 = results_1['Category'].value_counts()
    print("\nCounts by Category:")
    print(category_counts_1)
    
    print("\nDetailed Combination Counts:")
    for category in ['High Relevance Labels', 'Low Relevance Labels', 'Irrelevant Labels', 'Other']:
        if category in results_1['Category'].values:
            cat_df = results_1[results_1['Category'] == category]
            print(f"\n{category}:")
            combo_counts = cat_df['Labels'].value_counts()
            for combo, count in combo_counts.items():
                print(f"  {combo}: {count}")
    
    results_1.to_csv('label_combinations_1_match.csv', index=False)
    print("\n✓ Results saved to 'label_combinations_1_match.csv'")

# Create a summary table across all match counts
print("\n" + "="*80)
print("OVERALL SUMMARY TABLE")
print("="*80)

summary_data = []

for n_matches, results_df in [(3, results_3), (2, results_2), (1, results_1)]:
    if len(results_df) > 0:
        for category in ['High Relevance Labels', 'Low Relevance Labels', 'Irrelevant Labels', 'Other']:
            count = len(results_df[results_df['Category'] == category])
            summary_data.append({
                'Correct Labels': f'{n_matches} Correct Labels',
                'Category': category,
                'Count': count
            })

summary_table = pd.DataFrame(summary_data)
summary_pivot = summary_table.pivot(index='Correct Labels', columns='Category', values='Count').fillna(0).astype(int)
print(summary_pivot)

Number of Origins with exactly 3 TRUE matches: 467
Number of Origins with exactly 2 TRUE matches: 404
Number of Origins with exactly 1 TRUE match: 419

ANALYZING ORIGINS WITH 3 TRUE MATCHES

Counts by Category:
Other                    4155
High Relevance Labels    2675
Low Relevance Labels     1085
Irrelevant Labels         659
Name: Category, dtype: int64

Detailed Combination Counts:

High Relevance Labels:
  ('High', 'High', 'High'): 947
  ('High', 'High', 'Low'): 883
  ('High', 'Low', 'Low'): 773
  ('High', 'High', 'Irr'): 72

Low Relevance Labels:
  ('Low', 'Low', 'Low'): 894
  ('High', 'Irr', 'Low'): 191

Irrelevant Labels:
  ('Irr', 'Low', 'Low'): 443
  ('Irr', 'Irr', 'Low'): 131
  ('Irr', 'Irr', 'Irr'): 47
  ('High', 'Irr', 'Irr'): 38

Other:
  ('not applicable', 'not applicable', 'not applicable'): 3924
  ('High', 'Low', 'not applicable'): 47
  ('High', 'High', 'not applicable'): 46
  ('Irr', 'Low', 'not applicable'): 37
  ('Low', 'Low', 'not applicable'): 34
  ('Irr', 'Irr',

## Merge Trainee + Physician

In [15]:
# Load the data
trainee_df = pd.read_csv("Text Relevance Analysis Case View_052925.xlsx - Full Results 052925.csv")
print(trainee_df.columns)

physician_df = pd.read_csv("Physician_Final_Labels.csv")
print(physician_df.columns)

correct_answer_path = "gpt5-relevancy-combined-dec-12.csv"
correct_df = pd.read_csv(correct_answer_path)
# correct_df.columns

Index(['Origin', 'Problem ID', 'Duration', 'Labeling state',
       '# Qualified reads', 'User ID', 'q1', 'q2', 'q3', 'q4', 'q5', 'q6',
       'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13', 'q14', 'q15', 'q16',
       'q17', 'q18', 'q19', 'q20', 'q21', 'q22'],
      dtype='object')
Index(['Origin', 'Problem ID', 'Labeling state', 'User ID', 'q1', 'q2', 'q3',
       'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13', 'q14',
       'q15', 'q16', 'q17', 'q18', 'q19', 'q20', 'q21', 'q22'],
      dtype='object')


In [28]:
print(len(trainee_df))
print(len(physician_df))

7136
1204


In [16]:
trainee_physician_df = pd.concat([trainee_df, physician_df], ignore_index=True)

In [18]:
# Merge the dataframes on Origin and ID_corr
merged = trainee_physician_df.merge(
    correct_df[['ID_corr', 'answer_corr', 'data_source_corr']], 
    left_on='Origin', 
    right_on='ID_corr', 
    how='left'
)

# Add the answer_corr column to df
trainee_physician_df['answer_corr'] = merged['answer_corr']
trainee_physician_df['data_source_corr'] = merged['data_source_corr']

# Compare q1 with answer_corr (both converted to lowercase)
# If answer_corr is NaN (no match found), set Match to FALSE
trainee_physician_df['Match'] = merged.apply(
    lambda row: 'TRUE' if pd.notna(row['answer_corr']) and 
                str(row['q1']).lower() == str(row['answer_corr']).lower() 
                else 'FALSE', 
    axis=1
)

In [20]:
trainee_physician_df

,Origin,Problem ID,Duration,Labeling state,# Qualified reads,User ID,q1,q2,q3,q4,...,q16,q17,q18,q19,q20,q21,q22,answer_corr,data_source_corr,Match
0,ID2000,25261945,84.593,Labeled,3.0,579501,c,high relevance,high relevance,high relevance,...,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,NaN,NaN,FALSE
1,ID2000,25261945,180.905,Labeled,3.0,579796,d,low relevance,high relevance,high relevance,...,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,NaN,NaN,FALSE
2,ID2000,25261945,72.242,Labeled,3.0,581648,a,high relevance,high relevance,high relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,FALSE
3,ID1999,25261946,292.634,Labeled,4.0,579918,e,low relevance,high relevance,low relevance,...,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,C,medxpert,FALSE
4,ID1999,25261946,203.529,Labeled,4.0,581304,e,high relevance,high relevance,low relevance,...,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,C,medxpert,FALSE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8335,ID1013,25263932,NaN,Labeled,NaN,423438,d,high relevance,low relevance,low relevance,...,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,D,mmlu,TRUE
8336,ID1009,25263936,NaN,Labeled,NaN,423438,d,high relevance,low relevance,low relevance,...,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,D,medxpert,TRUE
8337,ID1005,25263940,NaN,Labeled,NaN,615169,c,high relevance,low relevance,low relevance,...,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,C,jama,TRUE
8338,ID1002,25263943,NaN,Labeled,NaN,24708,c,high relevance,high relevance,high relevance,...,not relevant,not applicable,not applicable,not applicable,not applicable,not applicable,not applicable,C,jama,TRUE


In [23]:
# Count unique Origins
unique_origins = trainee_physician_df['Origin'].nunique()
print(f"Number of unique Origins: {unique_origins}")
print("\n" + "="*50 + "\n")

# Summary: Count how many Origins have >=3, 2, 1, or 0 TRUE matches
print("="*50)
print("SUMMARY OF TRUE MATCHES PER ORIGIN:")
print("="*50)

true_counts = trainee_physician_df[trainee_physician_df['Match'] == 'TRUE'].groupby('Origin').size()
all_origins = trainee_physician_df.groupby('Origin').size()

# Count origins by number of TRUE matches (>=3 grouped together)
origins_with_3plus_true = (true_counts >= 3).sum()
origins_with_2_true = (true_counts == 2).sum()
origins_with_1_true = (true_counts == 1).sum()
origins_with_0_true = len(all_origins) - len(true_counts)

print(f"Origins with >=3 TRUE matches: {origins_with_3plus_true}")
print(f"Origins with 2 TRUE matches: {origins_with_2_true}")
print(f"Origins with 1 TRUE match: {origins_with_1_true}")
print(f"Origins with 0 TRUE matches: {origins_with_0_true}")

# Average total matches for Origins with >0 TRUE matches
print("\n" + "="*50)
print("AVERAGE TOTAL MATCHES FOR ORIGINS WITH >0 TRUE MATCHES:")
print("="*50)

origins_with_true = true_counts[true_counts > 0].index
total_matches_per_origin = all_origins[origins_with_true]
avg_total_matches = total_matches_per_origin.mean()

print(f"\nOverall average total matches (for Origins with >0 TRUE): {avg_total_matches:.2f}")

# Breakdown by data_source_corr
print("\nBreakdown by data_source_corr:")
print("-" * 50)

for data_source in trainee_physician_df['data_source_corr'].unique():
    df_source = trainee_physician_df[trainee_physician_df['data_source_corr'] == data_source]
    true_counts_source = df_source[df_source['Match'] == 'TRUE'].groupby('Origin').size()
    all_origins_source = df_source.groupby('Origin').size()
    origins_with_true_source = true_counts_source[true_counts_source > 0].index
    
    if len(origins_with_true_source) > 0:
        total_matches_source = all_origins_source[origins_with_true_source]
        avg_total_source = total_matches_source.mean()
        print(f"  {data_source}: {avg_total_source:.2f} (n={len(origins_with_true_source)} Origins)")
    else:
        print(f"  {data_source}: No Origins with TRUE matches")

# TRUE match counts breakdown by data_source_corr
print("\n" + "="*80)
print("TRUE MATCH COUNTS BREAKDOWN BY data_source_corr:")
print("="*80)

data_sources = []
origins_3plus_true_list = []
origins_2_true_list = []
origins_1_true_list = []
total_origins_list = []

for data_source in trainee_physician_df['data_source_corr'].unique():
    df_source = trainee_physician_df[trainee_physician_df['data_source_corr'] == data_source]
    true_counts_source = df_source[df_source['Match'] == 'TRUE'].groupby('Origin').size()
    all_origins_source = df_source.groupby('Origin').size()
    
    origins_3plus_true = (true_counts_source >= 3).sum()
    origins_2_true = (true_counts_source == 2).sum()
    origins_1_true = (true_counts_source == 1).sum()
    total_origins = len(all_origins_source)
    
    data_sources.append(data_source)
    origins_3plus_true_list.append(origins_3plus_true)
    origins_2_true_list.append(origins_2_true)
    origins_1_true_list.append(origins_1_true)
    total_origins_list.append(total_origins)

summary_table = pd.DataFrame({
    'Data Source': data_sources,
    '>=3 TRUE Matches': origins_3plus_true_list,
    '2 TRUE Matches': origins_2_true_list,
    '1 TRUE Match': origins_1_true_list,
    'Total Origins': total_origins_list
})

print(summary_table.to_string(index=False))

print("\n" + "-"*80)
totals = summary_table[['>=3 TRUE Matches', '2 TRUE Matches', '1 TRUE Match', 
                        'Total Origins']].sum()
print(f"{'TOTAL':<20} {totals['>=3 TRUE Matches']:>17} {totals['2 TRUE Matches']:>15} "
      f"{totals['1 TRUE Match']:>13} {totals['Total Origins']:>14}")

Number of unique Origins: 2000


SUMMARY OF TRUE MATCHES PER ORIGIN:
Origins with >=3 TRUE matches: 868
Origins with 2 TRUE matches: 340
Origins with 1 TRUE match: 92
Origins with 0 TRUE matches: 700

AVERAGE TOTAL MATCHES FOR ORIGINS WITH >0 TRUE MATCHES:

Overall average total matches (for Origins with >0 TRUE): 3.96

Breakdown by data_source_corr:
--------------------------------------------------
  nan: No Origins with TRUE matches
  medxpert: 3.82 (n=318 Origins)
  medbullets: 4.00 (n=207 Origins)
  mmlu: 4.07 (n=193 Origins)
  jama: 3.97 (n=582 Origins)

TRUE MATCH COUNTS BREAKDOWN BY data_source_corr:
Data Source  >=3 TRUE Matches  2 TRUE Matches  1 TRUE Match  Total Origins
        NaN                 0               0             0              0
   medxpert               107             149            62            318
 medbullets               168              35             4            207
       mmlu               181              12             0            193
       ja